In [1]:
import os
import pickle
import plotly.io as pio
pio.renderers.default = "notebook_connected"
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

In [3]:
# Charger les données

data_path = "../data/processed"
file_name = 'dataset_final.pkl'
with open(os.path.join(data_path, file_name), 'rb') as f:
    df = pickle.load(f)

In [4]:
# Charger un seul embedding

embeddings_path = "../data/embeddings"
# file_name = 'emb_sbert_multi.pkl'

# with open(os.path.join(embeddings_path, file_name), 'rb') as f:
#     embedding = pickle.load(f)

In [5]:
# Charger tous les embeddings

# embeddings_path = "../data/embeddings"
# embeddings = {}

# for file_name in os.listdir(embeddings_path):
#     with open(os.path.join(embeddings_path, file_name), 'rb') as f:
#         emb_name = file_name.split('.')[0]
#         embeddings[emb_name] = pickle.load(f)

### BERTopic

In [6]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
import nltk
from nltk.corpus import stopwords

/mnt/c/Users/charb/Documents/ALTERNANCE/Datascientest/Projet/TrustPilot/DS/trustpilot/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning:

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html



In [109]:
sentences = df["clean_comment"].tolist()

file_name_mutli = 'emb_sbert_multi.pkl'
with open(os.path.join(embeddings_path, file_name_mutli), 'rb') as f:
    embedding_multi = pickle.load(f)

file_name_fr = 'emb_sbert_fr.pkl'
with open(os.path.join(embeddings_path, file_name_fr), 'rb') as f:
    embedding_fr = pickle.load(f)

file_name_perf = 'emb_sbert_multi2.pkl'
with open(os.path.join(embeddings_path, file_name_fr), 'rb') as f:
    embedding_multi2 = pickle.load(f)

In [114]:
models = {}

for embedding, name in zip([embedding_fr, embedding_multi, embedding_multi2], ["français", "multilingue", "multilingue_2"]):

    vectorizer_model = CountVectorizer(
        stop_words=stopwords.words("french"),
        ngram_range=(1, 3),
        #min_df=2,
        max_df=0.95
    )
    
    topic_model = BERTopic(
        language="french",
        vectorizer_model=vectorizer_model,
        verbose=True,
        nr_topics="auto"
    )
    
    topics, probs = topic_model.fit_transform(sentences, embedding_multi)

    topic_model.reduce_topics(sentences, nr_topics=20)

    models[name] = (topic_model)

2025-11-17 18:43:44,738 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-11-17 18:43:47,674 - BERTopic - Dimensionality - Completed ✓
2025-11-17 18:43:47,676 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-11-17 18:43:48,088 - BERTopic - Cluster - Completed ✓
2025-11-17 18:43:48,090 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2025-11-17 18:43:52,173 - BERTopic - Representation - Completed ✓
2025-11-17 18:43:52,184 - BERTopic - Topic reduction - Reducing number of topics
2025-11-17 18:43:52,221 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-11-17 18:43:56,347 - BERTopic - Representation - Completed ✓
2025-11-17 18:43:56,360 - BERTopic - Topic reduction - Reduced number of topics from 159 to 18
2025-11-17 18:43:56,559 - BERTopic - Topic reduction - Reducing number of topics
2025-11-17 18:43:56,560 - BERTopic - Topic reduction - Number of topics (20) is equal or

### Evaluation

#### Diversité

In [115]:
from itertools import chain

def topic_diversity(topic_model, top_n=10):
    """
    Calcule la diversité des topics d'un modèle BERTopic.
    """
    topics = topic_model.get_topics()

    # On extrait les mots uniquement (sans les scores)
    topic_words = []
    for topic_id, word_scores in topics.items():
        # BERTopic place les topics -1 et autres meta-topics, donc on ignore topic -1
        if topic_id == -1:
            continue
        top_words = [w for (w, score) in word_scores[:top_n]]
        topic_words.append(top_words)

    # Liste aplatie
    all_words = list(chain.from_iterable(topic_words))
    unique_words = set(all_words)

    return len(unique_words) / len(all_words)


# Calcul du score pour chaque modèle
diversity_scores = {}

for name, model in models.items():
    score = topic_diversity(model, top_n=10)
    diversity_scores[name] = score

diversity_scores

{'français': 0.8764705882352941,
 'multilingue': 0.8631578947368421,
 'multilingue_2': 0.8263157894736842}

#### Score de cohérence

In [116]:
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def coherence_score(topic_model, documents, top_n=10):
    """
    Calcule le score de cohérence classique c_v pour un modèle BERTopic.
    """
    # Récupère les topics (liste de mots par topic)
    topics = topic_model.get_topics()
    topic_words = [
        [w for w, score in word_scores[:top_n]]
        for topic_id, word_scores in topics.items()
        if topic_id != -1
    ]

    # Préparer le corpus pour Gensim
    tokenized_docs = [doc.lower().split() for doc in documents]  # tokenisation simple
    dictionary = Dictionary(tokenized_docs)
    corpus = [dictionary.doc2bow(text) for text in tokenized_docs]

    # Calcul du score
    cm = CoherenceModel(
        topics=topic_words,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )
    
    return cm.get_coherence()

# Exemple
for name, model in models.items():
    score = coherence_score(model, sentences, top_n=10)
    print(f"{name} - Coherence c_v: {score:.4f}")

français - Coherence c_v: 0.4416
multilingue - Coherence c_v: 0.4632
multilingue_2 - Coherence c_v: 0.4920


#### Embedding-based coherence score

In [117]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def embedding_coherence(topic_model, embedder, top_n=10):
    """
    Calcule la cohérence basée sur les embeddings pour un modèle BERTopic.
    embedder : modèle sentence-transformers pour transformer les mots en vecteurs.
    """
    topics = topic_model.get_topics()
    scores = []

    for topic_id, word_scores in topics.items():
        if topic_id == -1:
            continue
        top_words = [w for w, _ in word_scores[:top_n]]
        word_embeddings = embedder.encode(top_words)
        sim_matrix = cosine_similarity(word_embeddings)
        
        # On enlève la diagonale (sim = 1)
        n = len(top_words)
        if n > 1:
            sims = (sim_matrix.sum() - n) / (n*(n-1))  # moyenne des cosinus
            scores.append(sims)

    return np.mean(scores)

# Exemple avec un modèle SentenceTransformer
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer('all-MiniLM-L6-v2')

for name, model in models.items():
    score = embedding_coherence(model, embedder, top_n=10)
    print(f"{name} - Embedding-based coherence: {score:.4f}")

français - Embedding-based coherence: 0.3266
multilingue - Embedding-based coherence: 0.3620
multilingue_2 - Embedding-based coherence: 0.3665


In [118]:
for name, model in models.items():
    with open(f"../models/topic_modeling/bertopic_{name}.pkl", "wb") as f:
        pickle.dump(model, f)

### Visualisation

In [120]:
# CHARGER LES MODELES ENREGISTRES
models = {}
for name in ["français", "multilingue", "multilingue_2"]:
    with open(os.path.join("../models/topic_modeling/", f"bertopic_{name}.pkl"), 'rb') as f:
        models[name] = pickle.load(f)

In [121]:
# topics trouvés
models["français"].get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,6533,-1_commande_plus_livraison_colis,"[commande, plus, livraison, colis, très, site,...","[bonjour à tous , un retour d ’ expérience con..."
1,0,7881,0_commande_très_livraison_plus,"[commande, très, livraison, plus, colis, bien,...",[oh vente privée mon cher ami vente privée dep...
2,1,227,1_montre_boucles_oreilles_boucles oreilles,"[montre, boucles, oreilles, boucles oreilles, ...","[bonjour , je commande une montre christian la..."
3,2,79,2_couleur_couleurs_photo_déçue,"[couleur, couleurs, photo, déçue, articles, dé...","[un peu déçue par la couleur ., j ai commander..."
4,3,56,3_bracelet_bracelets_très_déçue,"[bracelet, bracelets, très, déçue, plus, petit...",[bracelet connecté sur showroomprivé de la mar...
5,4,55,4_iphone_reconditionné_iphone reconditionné_té...,"[iphone, reconditionné, iphone reconditionné, ...","[je dois mettre une étoile , mais ça ne les va..."
6,5,43,5_cadeau_merci_très_signe,"[cadeau, merci, très, signe, fidélité, pendent...","[merci beaucoup pour ce cadeau, merci pour le ..."
7,6,31,6_robot_robots_aspirateur_toujours,"[robot, robots, aspirateur, toujours, aspirate...","[bonjour , j'ai renvoyé à showroomprivé un rob..."
8,7,29,7_baskets_taille_basket_paire,"[baskets, taille, basket, paire, commande, reç...",[j'ai remarqué de jolies baskets new balance e...
9,8,28,8_nickel_tout nickel_nickel rien_rien,"[nickel, tout nickel, nickel rien, rien, produ...","[ras nickel rien à dire, service nickel , aprè..."


In [122]:
# topics trouvés
models["multilingue"].get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,6442,-1_commande_plus_livraison_colis,"[commande, plus, livraison, colis, très, servi...",[très déçue du site vente privée ... bonne cli...
1,0,7856,0_commande_très_livraison_plus,"[commande, très, livraison, plus, site, colis,...","[bonjour , imaginez que cela fait un moment qu..."
2,1,134,1_taille_petit_trop_tailles,"[taille, petit, trop, tailles, grand, trop pet...",[alors mon expérience sur vente privé est la s...
3,2,87,2_montre_site_commandé montre_montres,"[montre, site, commandé montre, montres, plus,...","[déçue par la montre ., bonjour , j'ai acheté ..."
4,3,65,3_reçu_reçu colis_colis_rien reçu,"[reçu, reçu colis, colis, rien reçu, jamais, e...","[je n ’ ai pas reçu mon colis, je n ai pas reç..."
5,4,56,4_boucles_oreilles_boucles oreilles_oreille,"[boucles, oreilles, boucles oreilles, oreille,...",[boucles d'oreilles correspondant à mes attent...
6,5,55,5_bracelet_bracelets_très_déçue,"[bracelet, bracelets, très, déçue, déçue car, ...",[bracelet connecté sur showroomprivé de la mar...
7,6,54,6_iphone_reconditionné_iphone reconditionné_té...,"[iphone, reconditionné, iphone reconditionné, ...","[je dois mettre une étoile , mais ça ne les va..."
8,7,52,7_crème_cheveux_parfum_masque,"[crème, cheveux, parfum, masque, shampooing, m...",[j'ai acheté un lot de produits pour cheveux d...
9,8,49,8_cadeau_merci_fille_très,"[cadeau, merci, fille, très, ravie, fille ravi...","[merci beaucoup pour ce cadeau, merci pour le ..."


In [28]:
# Mots-clés associés à un topic
models["français"].get_topic(0)

[('commande', np.float64(0.025884285984158574)),
 ('livraison', np.float64(0.025211979046278556)),
 ('très', np.float64(0.024817338316522306)),
 ('plus', np.float64(0.020921395958509892)),
 ('colis', np.float64(0.018338502160680822)),
 ('bien', np.float64(0.018071155498876515)),
 ('service', np.float64(0.01718369697262888)),
 ('site', np.float64(0.016815205010169126)),
 ('qualité', np.float64(0.01647076409897316)),
 ('tout', np.float64(0.015404110809493402))]

In [29]:
# Mots-clés associés à un topic
models["multilingue"].get_topic(0)

[('commande', np.float64(0.01334156410495559)),
 ('livraison', np.float64(0.012767621325075182)),
 ('très', np.float64(0.011153080393126093)),
 ('plus', np.float64(0.010639825498328792)),
 ('service', np.float64(0.009334400229026532)),
 ('colis', np.float64(0.008835134628431855)),
 ('bien', np.float64(0.00879317252851676)),
 ('client', np.float64(0.008724232335621123)),
 ('site', np.float64(0.008699768087124105)),
 ('rien', np.float64(0.008514388082079043))]

In [30]:
fig1 = models["français"].visualize_topics(top_n_topics=10)
fig2 = models["multilingue"].visualize_topics(top_n_topics=10)

combined_fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Modèle 1 : Français", "Modèle 2 : Multilingue")
)

for trace in fig1['data']:
    combined_fig.add_trace(trace, row=1, col=1)

for trace in fig2['data']:
    combined_fig.add_trace(trace, row=1, col=2)

combined_fig.update_layout(
    title_text="Répartition des topics",
    showlegend=False,
    height=600,
    width=1000
)

combined_fig.show()

In [31]:
models["français"].visualize_barchart(top_n_topics=10)

In [32]:
models["multilingue"].visualize_barchart(top_n_topics=10)

In [33]:
models["multilingue"].visualize_hierarchy()

In [34]:
models["multilingue"].visualize_hierarchy()